# Afri-work & Hahu Jobs Data Scraper

This notebook fetches job listings from:
1. **Afri-work** – all jobs (anonymous)
2. **Afri-work** – software/tech jobs only (filtered by sector)
3. **Hahu Jobs** – tech sector jobs

Results are saved as CSV files. Optionally mount Google Drive to persist the files.

## 1. Install / import dependencies

`requests` and `pandas` are pre-installed in Google Colab, so no extra install is needed.

In [16]:
import requests
import pandas as pd
import json
import os
from datetime import datetime

print(f"pandas {pd.__version__}  |  requests {requests.__version__} kjlkjljlkjlkj")

pandas 2.2.2  |  requests 2.32.4 kjlkjljlkjlkj


## 2. (Optional) Mount Google Drive

Run this cell to save output CSV files directly to your Google Drive.
Skip it if you only need the files in the Colab session.

In [17]:
USE_GOOGLE_DRIVE = False  # Set to False to skip Drive mounting

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/job_data'
else:
    OUTPUT_DIR = '/content/job_data'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Output directory: /content/job_data


## 3. Helper utilities

In [18]:
def run_graphql_query(url: str, query: str, variables: dict, headers: dict) -> dict:
    """Send a GraphQL POST request and return the parsed JSON response."""
    payload = {"query": query, "variables": variables}
    response = requests.post(url, headers=headers, json=payload, timeout=60)
    response.raise_for_status()
    data = response.json()
    if "errors" in data:
        raise RuntimeError(f"GraphQL errors: {data['errors']}")
    return data


def save_csv(df: pd.DataFrame, filename: str) -> str:
    """Save a DataFrame to a CSV file and return the full path."""
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False)
    print(f"  Saved {len(df)} rows → {path}")
    return path


def timestamp_filename(prefix: str) -> str:
    """Return a filename like 'prefix_YYYYMMDD_HHMMSS.csv'."""
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{prefix}_{ts}.csv"

## 4. Afri-work – all jobs (anonymous)

Uses the public, unauthenticated endpoint. No credentials required.

In [19]:
AFRIWORK_URL = "https://api.afriworket.com/v1/graphql"

AFRIWORK_HEADERS_ANON = {
    "accept": "application/graphql-response+json, application/graphql+json, application/json, text/event-stream, multipart/mixed",
    "accept-language": "en-US,en;q=0.5",
    "authorization": "Bearer undefined",
    "content-type": "application/json",
    "origin": "https://afriworket.com",
    "referer": "https://afriworket.com/",
    "x-hasura-role": "anonymous",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36",
}

AFRIWORK_QUERY = """
query GetAllJobs($offset: Int!, $whereCondition: jobs_bool_exp!, $orderCondition: [jobs_order_by!]) {
  jobs(
    order_by: $orderCondition
    offset: $offset
    where: $whereCondition
  ) {
    id
    approval_status
    title
    created_at
    published_at
    refreshed_at
    description
    job_type
    job_site
    skill_requirements {
      skill {
        name
        id
      }
    }
    city {
      name
      country {
        name
      }
    }
    sectors {
      sector {
        name
        id
      }
    }
    deadline
    compensation_amount_cents
    compensation_type
    compensation_currency
    experience_level
    entity {
      type
      name
    }
  }
}
"""

AFRIWORK_VARIABLES_ALL = {
    "offset": 0,
    "orderCondition": {"published_at": "desc"},
    "whereCondition": {},
}

In [20]:
def flatten_afriwork_job(job: dict) -> dict:
    """Flatten a single Afri-work job record into a plain dict."""
    skills = ", ".join(
        sr["skill"]["name"]
        for sr in (job.get("skill_requirements") or [])
        if sr.get("skill")
    )
    sectors = ", ".join(
        s["sector"]["name"]
        for s in (job.get("sectors") or [])
        if s.get("sector")
    )
    city_name = (job.get("city") or {}).get("name", "")
    country_name = ((job.get("city") or {}).get("country") or {}).get("name", "")
    entity_name = (job.get("entity") or {}).get("name", "")
    entity_type = (job.get("entity") or {}).get("type", "")

    return {
        "id": job.get("id"),
        "title": job.get("title"),
        "approval_status": job.get("approval_status"),
        "job_type": job.get("job_type"),
        "job_site": job.get("job_site"),
        "experience_level": job.get("experience_level"),
        "deadline": job.get("deadline"),
        "published_at": job.get("published_at"),
        "created_at": job.get("created_at"),
        "refreshed_at": job.get("refreshed_at"),
        "compensation_amount_cents": job.get("compensation_amount_cents"),
        "compensation_type": job.get("compensation_type"),
        "compensation_currency": job.get("compensation_currency"),
        "city": city_name,
        "country": country_name,
        "entity_name": entity_name,
        "entity_type": entity_type,
        "sectors": sectors,
        "skills": skills,
        "description": job.get("description"),
    }


print("Fetching all Afri-work jobs…")
aw_all_data = run_graphql_query(
    AFRIWORK_URL,
    AFRIWORK_QUERY,
    AFRIWORK_VARIABLES_ALL,
    AFRIWORK_HEADERS_ANON,
)
aw_all_jobs = aw_all_data["data"]["jobs"]
print(f"  Retrieved {len(aw_all_jobs)} jobs")

df_aw_all = pd.DataFrame([flatten_afriwork_job(j) for j in aw_all_jobs])
save_csv(df_aw_all, timestamp_filename("afriwork_all_jobs"))
df_aw_all.head(3)

Fetching all Afri-work jobs…
  Retrieved 53122 jobs
  Saved 53122 rows → /content/job_data/afriwork_all_jobs_20260425_172759.csv


,id,title,approval_status,job_type,job_site,experience_level,deadline,published_at,created_at,refreshed_at,compensation_amount_cents,compensation_type,compensation_currency,city,country,entity_name,entity_type,sectors,skills,description
0,7b30c980-2601-4650-8d8e-8d7149d6966a,Intern React Developer,EXPIRED,PAID_INTERN,Unassigned,INTERMEDIATE,None,None,2023-05-02T12:21:08.391921+00:00,None,NaN,MONTHLY,None,ADDIS ABABA,ETHIOPIA,Natnael Mekonnen,private_client,Software Design & Development,,Job Description: We are currently seeking a hi...
1,ef3a08e5-6b4f-4f74-80d2-7e37931f6782,Animal science,EXPIRED,FULL_TIME,ONSITE,JUNIOR,2024-02-13T00:00:00+00:00,None,2024-02-13T16:08:07.515102+00:00,None,NaN,MONTHLY,None,ADDIS ABABA,ETHIOPIA,Negese Getu,private_client,Journalism & Communication,,I am interested to join your company and I am ...
2,ab072d2d-3182-4184-bb34-e0ea5a64c061,Hotel Front Desk Receptionist,PUBLISHED,FULL_TIME,ONSITE,JUNIOR,2026-05-09T23:59:59+00:00,2026-04-25T14:16:44.544+00:00,2026-04-25T13:37:28.887831+00:00,None,NaN,MONTHLY,None,ADDIS ABABA,ETHIOPIA,Michael Aklilu,private_client,Customer Service & Care,Communication,"<p>We are seeking an <strong>energetic, and ac..."


In [21]:
df_aw_all.tail(3)

,id,title,approval_status,job_type,job_site,experience_level,deadline,published_at,created_at,refreshed_at,compensation_amount_cents,compensation_type,compensation_currency,city,country,entity_name,entity_type,sectors,skills,description
53119,247caa65-7fe7-49f6-9b97-78c473dc0309,Oromiffa Localization and Translation expert,CLOSED,FREELANCE,Unassigned,INTERMEDIATE,2022-08-26T15:57:46.299+00:00,2022-08-12T15:57:46.299+00:00,2022-08-12T15:47:46.0971+00:00,None,500000.0,MONTHLY,ETB,ADDIS ABABA,ETHIOPIA,Semegn Tadesse,private_client,Journalism & Communication,,We are looking for an expert translator for or...
53120,5a7b5862-c749-44f7-aa7c-6241e75f7522,.Net .NetCore Blazer Developer,CLOSED,FREELANCE,REMOTE,INTERMEDIATE,2022-08-26T06:38:10.402+00:00,2022-08-12T06:38:10.402+00:00,2022-08-12T06:26:19.160149+00:00,None,NaN,MONTHLY,None,ADDIS ABABA,ETHIOPIA,Semegn Tadesse,private_client,Software Design & Development,,We are looking for an expert .Net .NetCore Bla...
53121,81b7df5e-ccc7-4a02-a509-d10177f0d1d7,.Net .NetCore Blazer Developer,CLOSED,FREELANCE,Unassigned,INTERMEDIATE,2022-08-25T20:35:16.756+00:00,2022-08-11T20:35:16.756+00:00,2022-08-11T20:12:41.315928+00:00,None,NaN,MONTHLY,None,ADDIS ABABA,ETHIOPIA,Semegn Tadesse,private_client,Software Design & Development,,We are looking for an expert .Net .NetCore Bla...


## 5. Afri-work – Software/Tech jobs (filtered)

This endpoint can be accessed anonymously (no Bearer token required).
If you have a personal Bearer token from your Afri-work account you can
paste it below to access the `job_seeker` role which may return additional fields.

In [22]:
# ---------------------------------------------------------------
# Optional: paste your own Bearer token here (leave as-is to use
# the anonymous role which does not require authentication).
# ---------------------------------------------------------------
AFRIWORK_BEARER_TOKEN = ""  # e.g. "eyJhbGciOiJ..."

if AFRIWORK_BEARER_TOKEN:
    AFRIWORK_HEADERS_TECH = {
        **AFRIWORK_HEADERS_ANON,
        "authorization": f"Bearer {AFRIWORK_BEARER_TOKEN}",
        "x-hasura-role": "job_seeker",
    }
else:
    # Fall back to anonymous role
    AFRIWORK_HEADERS_TECH = AFRIWORK_HEADERS_ANON

AFRIWORK_VARIABLES_TECH = {
    "offset": 0,
    "orderCondition": {"published_at": "desc"},
    "whereCondition": {
        "sectors": {
            "sector": {
                "name": {"_eq": "Software Design & Development"}
            }
        }
    },
}

print("Fetching Afri-work Software/Tech jobs…")
aw_tech_data = run_graphql_query(
    AFRIWORK_URL,
    AFRIWORK_QUERY,
    AFRIWORK_VARIABLES_TECH,
    AFRIWORK_HEADERS_TECH,
)
aw_tech_jobs = aw_tech_data["data"]["jobs"]
print(f"  Retrieved {len(aw_tech_jobs)} tech jobs")

df_aw_tech = pd.DataFrame([flatten_afriwork_job(j) for j in aw_tech_jobs])
save_csv(df_aw_tech, timestamp_filename("afriwork_tech_jobs"))
df_aw_tech.head(3)

Fetching Afri-work Software/Tech jobs…
  Retrieved 4000 tech jobs
  Saved 4000 rows → /content/job_data/afriwork_tech_jobs_20260425_173200.csv


,id,title,approval_status,job_type,job_site,experience_level,deadline,published_at,created_at,refreshed_at,compensation_amount_cents,compensation_type,compensation_currency,city,country,entity_name,entity_type,sectors,skills,description
0,7b30c980-2601-4650-8d8e-8d7149d6966a,Intern React Developer,EXPIRED,PAID_INTERN,Unassigned,INTERMEDIATE,None,None,2023-05-02T12:21:08.391921+00:00,None,NaN,MONTHLY,None,ADDIS ABABA,ETHIOPIA,Natnael Mekonnen,private_client,Software Design & Development,,Job Description: We are currently seeking a hi...
1,517b0227-48fd-4dac-8242-519349c05c07,Full stack software developer,PUBLISHED,FULL_TIME,ONSITE,SENIOR,2026-04-29T00:00:00+00:00,2026-04-24T10:48:47+00:00,2026-04-24T10:48:47.871656+00:00,None,2500000.0,MONTHLY,ETB,ADDIS ABABA,ETHIOPIA,POSSIBLE TECHNOLOGY PLC,company,Software Design & Development,"Software Maintenance, Software Development, So...",<p> Full Stack Developer (MERN &amp; Flutter)<...
2,730597f2-b116-43b5-99be-cdea11ea4943,Full stack software developer,CLOSED,FULL_TIME,ONSITE,SENIOR,2026-04-29T00:00:00+00:00,2026-04-24T10:47:53+00:00,2026-04-24T10:47:53.983046+00:00,None,2500000.0,MONTHLY,ETB,ADDIS ABABA,ETHIOPIA,POSSIBLE TECHNOLOGY PLC,company,Software Design & Development,,<p> Full Stack Developer (MERN &amp; Flutter)<...


## 6. Hahu Jobs – Tech sector (filtered)

In [28]:
HAHU_URL = "https://graph.aggregator.hahu.jobs/v1/graphql"

HAHU_HEADERS = {
    "Accept": "*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
    "Origin": "https://www.hahu.jobs",
    "Referer": "https://www.hahu.jobs/",
    "content-type": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
}

HAHU_QUERY = """
query ($args: search_jobs_args!, $filter: jobs_bool_exp, $limit: Int, $offset: Int, $orderBy: [jobs_order_by!]) {
  search_jobs_aggregate(args: $args, where: $filter) {
    aggregate {
      count
    }
  }
  jobs: search_jobs(
    where: $filter
    order_by: $orderBy
    args: $args
    offset: $offset
    limit: $limit
  ) {
    id
    title
    created_at
    total_web_view_count
    telegram_view_count
    total_view_count
    type
    max_years_of_experience
    years_of_experience
    summary
    salary
    deadline
    expired
    location
    source
    application_method
    application_url
    application_email
    number_of_applicants
    approved_on
    job_cities {
      city {
        name
        region {
          name
          id
        }
      }
    }
    entity {
      logo
      name
      id
    }
    sub_sector {
      name
      sector {
        name
        id
        icon_class
        icon_code
      }
    }
    area {
      address
      name
    }
    isco_08 {
      isco_08_code
      title_en
      title_am
    }
    soc_2010 {
      title
      onetsoc_code
    }
    esco_code
  }
}
"""

# ---------------------------------------------------------------
# Sector ID for IT / Tech on Hahu Jobs. 
# ---------------------------------------------------------------
HAHU_TECH_SECTOR_ID = "iL3qCg1STtXlr8fc_edq3"

HAHU_VARIABLES = {
    "limit": 10000,
    "filter": {
        "_and": [
            {
                "_and": [
                    {"years_of_experience": {"_gte": 0}},
                    {"years_of_experience": {"_lte": 10}},
                ]
            },
            {
                "sub_sector": {
                    "sector": {
                        "id": {"_eq": HAHU_TECH_SECTOR_ID}
                    }
                }
            }
        ]
    },
    "args": {},
    "offset": 0,
    "orderBy": [{"priority": "desc_nulls_last"}, {"deadline": "asc"}],
}

# ---------------------------------------------------------------
# Variables for fetching ALL jobs (no sector filter)
# ---------------------------------------------------------------
HAHU_ALL_JOBS_VARIABLES = {
    "limit": 10000,
    "filter": {
        "_and": [
            {
                "_and": [
                    {"years_of_experience": {"_gte": 0}},
                    {"years_of_experience": {"_lte": 10}},
                ]
            }
        ]
    },
    "args": {},
    "offset": 0,
    "orderBy": [{"priority": "desc_nulls_last"}, {"deadline": "asc"}],
}

In [29]:
def flatten_hahu_job(job: dict) -> dict:
    """Flatten a single Hahu Jobs record into a plain dict."""
    cities = ", ".join(
        jc["city"]["name"]
        for jc in (job.get("job_cities") or [])
        if jc.get("city")
    )
    regions = ", ".join(
        (jc["city"].get("region") or {}).get("name", "")
        for jc in (job.get("job_cities") or [])
        if jc.get("city")
    )
    entity_name = (job.get("entity") or {}).get("name", "")
    entity_logo = (job.get("entity") or {}).get("logo", "")
    sub_sector_name = (job.get("sub_sector") or {}).get("name", "")
    sector_name = ((job.get("sub_sector") or {}).get("sector") or {}).get("name", "")
    area_name = (job.get("area") or {}).get("name", "")
    area_address = (job.get("area") or {}).get("address", "")
    isco_code = (job.get("isco_08") or {}).get("isco_08_code", "")
    isco_title_en = (job.get("isco_08") or {}).get("title_en", "")
    soc_title = (job.get("soc_2010") or {}).get("title", "")
    soc_code = (job.get("soc_2010") or {}).get("onetsoc_code", "")

    return {
        "id": job.get("id"),
        "title": job.get("title"),
        "type": job.get("type"),
        "source": job.get("source"),
        "location": job.get("location"),
        "cities": cities,
        "regions": regions,
        "area_name": area_name,
        "area_address": area_address,
        "salary": job.get("salary"),
        "deadline": job.get("deadline"),
        "approved_on": job.get("approved_on"),
        "expired": job.get("expired"),
        "years_of_experience": job.get("years_of_experience"),
        "max_years_of_experience": job.get("max_years_of_experience"),
        "experience_level": job.get("experience_level") if "experience_level" in job else None,
        "application_method": job.get("application_method"),
        "application_url": job.get("application_url"),
        "application_email": job.get("application_email"),
        "number_of_applicants": job.get("number_of_applicants"),
        "total_view_count": job.get("total_view_count"),
        "total_web_view_count": job.get("total_web_view_count"),
        "telegram_view_count": job.get("telegram_view_count"),
        "entity_name": entity_name,
        "entity_logo": entity_logo,
        "sub_sector": sub_sector_name,
        "sector": sector_name,
        "isco_08_code": isco_code,
        "isco_08_title_en": isco_title_en,
        "soc_2010_title": soc_title,
        "soc_2010_code": soc_code,
        "esco_code": job.get("esco_code"),
        "summary": job.get("summary"),
    }


print("Fetching Hahu Jobs (tech sector)…")
hahu_data = run_graphql_query(HAHU_URL, HAHU_QUERY, HAHU_VARIABLES, HAHU_HEADERS)
hahu_jobs = hahu_data["data"]["jobs"]
total_count = hahu_data["data"]["search_jobs_aggregate"]["aggregate"]["count"]
print(f"  Retrieved {len(hahu_jobs)} jobs (aggregate count: {total_count})")

df_hahu = pd.DataFrame([flatten_hahu_job(j) for j in hahu_jobs])
save_csv(df_hahu, timestamp_filename("hahu_tech_jobs"))

print("Fetching ALL Hahu Jobs…")
hahu_all_data = run_graphql_query(HAHU_URL, HAHU_QUERY, HAHU_ALL_JOBS_VARIABLES, HAHU_HEADERS)
hahu_all_jobs = hahu_all_data["data"]["jobs"]
total_all_count = hahu_all_data["data"]["search_jobs_aggregate"]["aggregate"]["count"]
print(f"  Retrieved {len(hahu_all_jobs)} jobs (aggregate count: {total_all_count})")

df_hahu_all = pd.DataFrame([flatten_hahu_job(j) for j in hahu_all_jobs])
save_csv(df_hahu_all, timestamp_filename("hahu_all_jobs"))

df_hahu_all.head(3)

Fetching Hahu Jobs (tech sector)…
  Retrieved 43 jobs (aggregate count: 43)
  Saved 43 rows → /content/job_data/hahu_tech_jobs_20260425_173703.csv
Fetching ALL Hahu Jobs…
  Retrieved 1028 jobs (aggregate count: 1028)
  Saved 1028 rows → /content/job_data/hahu_all_jobs_20260425_173705.csv


,id,title,type,source,location,cities,regions,area_name,area_address,salary,...,entity_name,entity_logo,sub_sector,sector,isco_08_code,isco_08_title_en,soc_2010_title,soc_2010_code,esco_code,summary
0,G67EDBYYUEyFvGV1,Senior Office Engineer,full_time,hahujobs_enterprise,None,Addis Ababa,Addis Ababa,,,NaN,...,ATTA Trading PLC,https://cdn.hahu.jobs/public/aggregator/telegr...,Civil Engineering,Engineering,2142,Civil Engineers,Civil Engineers,17-2051.00,2142.1,Bachelor's Degree in Civil Engineering or with...
1,qati4VjlU7YtnFtU,Batching Plant Operator/Mechanic,full_time,hahujobs_enterprise,None,Addis Ababa,Addis Ababa,,,NaN,...,Ethiopian Engineering Corporation,https://cdn.hahu.jobs/public/aggregator/telegr...,Heavy Machinery Operation,Low and Medium Skilled Worker,8114,"Cement, Stone and Other Mineral Products Machi...","Plant and System Operators, All Other",51-8099.00,8114.3,ቲቪቲ ሌቭል 1 በሜካኒክስ ወይም በተመሳሳይ የትምህርት መስክ የተመረቀ/ች...
2,69e788fb46f9c112d7a228e7,Assistant Chef,full_time,hahujobs_telegram,None,Debre Markos,Amhara,,,NaN,...,Baye Atenafu General Trading,None,Hotels and Restaurants Services,Hospitality,3434,Chefs,Chefs and Head Cooks,35-1011.00,3434.1,"Certificate/Diploma in Food Preparation, Kitch..."


## 7. Summary

In [30]:
print("=" * 50)
print("Collection complete")
print(f"  Afri-work all jobs   : {len(df_aw_all):>6} rows")
print(f"  Afri-work tech jobs  : {len(df_aw_tech):>6} rows")
print(f"  Hahu tech jobs       : {len(df_hahu):>6} rows")
print(f"  Hahu all jobs        : {len(df_hahu_all):>6} rows")
print(f"  Output directory     : {OUTPUT_DIR}")
print("=" * 50)

Collection complete
  Afri-work all jobs   :  53122 rows
  Afri-work tech jobs  :   4000 rows
  Hahu tech jobs       :     43 rows
  Hahu all jobs        :   1028 rows
  Output directory     : /content/job_data
